# Traveloka Hotel Data Analytics

In [1]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
base_url = "https://www.traveloka.com/vi-vn/hotel/vietnam/region/ho-chi-minh-city-10009794"
driver.get(base_url)

## 6. Hotel List

In [ ]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

# Số trang cần cào 
page = 15
base_paging_url = "https://www.traveloka.com/vi-vn/hotel/vietnam/region/ho-chi-minh-city-10009794"
hotel_data = []

try:
    target_url = f"{base_paging_url}/{page}?viewType=list"
    print(f"\n--- Đang xử lý Trang {page} ---")
    
    driver.get(target_url)
    time.sleep(6)

    for i in range(4): 
        driver.execute_script("window.scrollBy(0, 1000);")
        print(f"   > Đang nạp dữ liệu vùng {i+1}...")
        time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    all_links = soup.find_all("a", href=re.compile(r'/hotel/.*-[0-9]+'))
    
    links_to_extract = all_links[:21] 
    print(f"   > Đã tìm thấy {len(all_links)} khách sạn. Chỉ bóc tách {len(links_to_extract)} cái đầu tiên.")

    for link in links_to_extract:
        href = link.get('href')
        match = re.search(r'-(\d+)(?:\?|$)', href)
        if match:
            h_id = match.group(1)
            h_name = link.find("h3").get_text(strip=True) if link.find("h3") else "N/A"
            hotel_data.append({"Hotel ID": h_id, "Name": h_name})
            
    if hotel_data:
        folder_path = "./data"
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"ã tự động tạo thư mục: {folder_path}")

        df_page = pd.DataFrame(hotel_data).drop_duplicates(subset=["Hotel ID"])
        output_path = f"{folder_path}/hotel_ids_page_{page}_limit21.csv"
        df_page.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        print(f"\n--- THÀNH CÔNG ---")
        print(f"Đã lưu {len(df_page)} ID khách sạn vào file: {output_path}")

except Exception as e:
    print(f"Lỗi: {e}")


--- Đang xử lý Trang 15 ---
   > Đang nạp dữ liệu vùng 1...
   > Đang nạp dữ liệu vùng 2...
   > Đang nạp dữ liệu vùng 3...
   > Đang nạp dữ liệu vùng 4...
   > Đã tìm thấy 205 khách sạn. Chỉ bóc tách 21 cái đầu tiên.
📁 Đã tự động tạo thư mục: ./data

--- THÀNH CÔNG ---
Đã lưu 21 ID khách sạn vào file: ./data/hotel_ids_page_15_limit21.csv


Crawl comment 

In [8]:
import requests
import pandas as pd
import time
import random
import os
from tqdm import tqdm

URL = "https://www.traveloka.com/api/v2/ugc/review/consumption/v2/getReviews"

COOKIES = """countryCode=VN; _fbp=fb.1.1774159974654.493892045273194278; _gcl_au=1.1.2083837330.1774159975; _tt_enable_cookie=1; _ttp=01KMA2SMMJ70YN0C0S4FPAK9Y0_.tt.1; _fwb=1243FlO8bEbxMrBsS1iM8if.1774159975077; _cs_ex=1760605804; _cs_c=1; _gid=GA1.2.1523385916.1774159975; __spdt=d3e9aca0b71e4fec8cb492a642008886; _yjsu_yjad=1774159975.48effbb1-f3cf-4359-b068-a704815eff45; _kmpid=km|www.traveloka.com|1774159975386|f7594e5e-a8e3-4f81-94d6-016bea8636bd; _kmpid=km|traveloka.com|1774159975386|f7594e5e-a8e3-4f81-94d6-016bea8636bd; _ly_su=1774159975.48effbb1-f3cf-4359-b068-a704815eff45; __lt__cid=1c6511dc-f22e-4e0a-aef9-095651a2813e; _pin_unauth=dWlkPVpUa3daV1ZsWkdNdE5UUTNZeTAwTnpGaExXRm1PVE10TlRaak5qbGpZelpsTnpobA; _im_vid=01KMA2SNAPB1D4STSMMRZ301ZC; tv_cs=1; tv-repeat-visit=true; __lt__sid=1097d2a3-f0fc4975; g_state={"i_l":0,"i_ll":1774267442045,"i_e":{"enable_itp_optimization":0},"i_b":"5j/zxLDvU7icR0r3lIJIq1Fvenh5dGcE2xeZvChqT30"}; tv_user={"authorizationLevel":"400","id":"405461192"}; clientSessionId=T1-web.01KMDBT82QE2C900SQPBXC2X2G; tv_mcc_id=01KMDCFMTFGC82RTFZGG5N9Q2Q; tv_lt=1774271837457; datadome=LiksKA9dWAcVmqXMoiYVreGtedackkvKHwCGiCd8GoFkdtV6xmObGzMFMyALeueazIi1blarCCWr~N0dghizwdADQ6pRjXBV2~4w_x66rtWBMqaMDs724tOi38Wn7EEd; aws-waf-token=d55ed907-d79b-49f7-adba-af81ce780a23:NQoArMZdH9cGAAAA:xl1sa4lsXCK+wdgq4UjLAj70qgq90vG1SFPGuF5QMEhSCoA1sRapb7sgmw6JQHenNitwJCfFpEIf1duhQXeRB48b4u4yrjdpoSuuQdXlQiu4Y3X0qjGikRfJvA84BrfORLkP/8b0MJ2pfS+dTvxA8zsoxWquBNaTl7fpaaKFPXWctfVxzReS7ABsj+FXlVDl+bo=; amp_f4354c=RpuPS_kWfuJ3AA00_JB6XW...1jkd990l7.1jkddqufh.0.8.8; __rtbh.lid=%7B%22eventType%22%3A%22lid%22%2C%22id%22%3A%22HG3mfpwLwNPIPFsm1MP5%22%2C%22expiryDate%22%3A%222027-03-23T13%3A23%3A36.057Z%22%7D; _gat_UA-29776811-12=1; _rdt_uuid=1774159975198.cfc9fabe-15dd-4ae0-8f77-8fffcbe0ae3d; wcs_bt=s_2cb982ada97c:1774272216; ttcsid_CFNI0BRC77UEUGLEG00G=1774270478111::VN_-yPZSL0QC6MKl3Veg.5.1774272216824.1; cto_bundle=k51ZHl9yemYzYiUyQjhUVFI0MW45Z0ZENnhUMUFTMCUyQjNZbWxBZTlub0pIRmNESWw2N1RNOWQlMkZ2Q3kyWGVpSmJvekNKMnU3VmtlaEJxZ2hpWFRIcjdhbjdrOVU3eDlyaGZtNktyZiUyQjhSY1k5QVpEJTJGTG5YZ29xeUg5STNNOWNWQXl2MVpuNVRobUxYaHJKMWpkclE0dEU4Sm9hcnl3JTNEJTNE; _ga=GA1.2.1998283078.1774159975; _ga_RSRSMMBH0X=GS2.1.s1774267434$o8$g1$t1774272217$j55$l1$h665420350; amp_1a5adb=eV7HmxqXsSvhnUq1tHrZSG...1jkd990l4.1jkddr0oc.fu.8.g6; __smn_fid=djSoYHvGEesPBGXNmGDdNAib28I1EMq0s6HRM0Dr-f2gkbnTFA; tvo=L2FwaS9zZW4vc3M=; tvl=qgdHX7GvehrD9XH5a3S4PUiOJGezXQ9yizVaSxTklwrLYY64AE4apiD1qmHRGaV8gGAQoV6xR5wi1hxtboYegx0JoHbuxL9J5IDMykh7yrkDV+tDU0tOQZDC0wf5Bz7Ih8SGYm0D03zEW7S7g02l9zkAPbkMGQ6AJj+0Bs51j2cIuvBWPTXZfAtgMLdYJtNBDc3j1JSGR5kV3Pzs98iB4lpuxJI/DBjHz/fMWezZ7kXZ7rnY/ZECKTFkPgC5XOoegPSmFWTORUTsShnZeSiZ3/yP7xMQPeKQopkkGD5gg8g3eH8y/xjSHQuqGSulUuZefhIvPYQKz9JN6XotHjyY4BicIcRda2on50AabPD1nMDE72Pp7hdRfTKY58kLOK/7HkY8+qNQt2lTq4wdBGjAaVzB+GOmQueNjMH1Si++CMLZwqpPhQGQOXwtSy7LciXbM83R2Okh4F/aBhsoUasw1AqOq2Fi99NmV31O2HgLT7Vi6megBHdUAfLyB+4W4NnD37wi24PxRoQ/bnNAmqp3d45ASUFgMpl3Vlu+aCTxzqBJ8/6qXkrr+RexL/BH3ktO; tvs=qgdHX7GvehrD9XH5a3S4PXWKx93/3Xi103f/kPpnhg1IQez7AjqOPow88qqCMiL7CqvJjpn5Z2svD8QZzAmUN8I7bk86ki9gpnRvB7n1SMiC7eTGHQ9qWr8bxlOoq3yfxCOR+j414vXaWkTN13y8Xm6TR9THuwP40xwr0KaBWhyuKLek0naSAb8wZSbMAqHC//kZSWHuxyRLSSbE1fyhbFewewR5PTnP1X9At/GV3QJMVeoaxEQheXCzmm0cEa6sU+2waoklnoesZu5l0Qs43FkhaqBMeH5nANhcsopPyCOI+bn/O3nobzjRm9doZvIajaIH7/PbO2DMHcytOwGWQmz0CHU5wjNkgJdR1wrPgApxzJesHXMQKKQRx9z8lOV3wdjTV5u6cw+F0z/9GQQY6xXsISkKDwfVWc1LzRa8ePq9dIjFJFgXSdafORq8l92INePF9eQ6XrHiVq/BzcnIG8CX0Y60ff/szwr54Bbvrncwu8DsdSoM/65tQvTQVRB16mwVRNPkqdf/FOIJsJIIuyjkVG2s9e19bYP/hdVerM6F7b91cWH7NdLN5aSJrJ+EzzZP+rzw3XjTV2HqIZyhOjq3pwdh2VE7UhxJPEakYU=; sen_t=Adt8GUKnR5OJ3StiVpTKR8iHvIFhPRc6dX7fROeyxjek+PdrQJyCgIsmxDhQ/6tRti/EfalDU42qwbJt8VJ53I9vDL4a/TDA07NhU0nUu2WdL4CUn2ZrurOSW3jS; _dd_s=rum=0&expire=1774273118838&logs=1&id=3e3f3aee-f187-433e-a223-2f3b93f60cbb&created=1774270064522; ttcsid=1774267434290::ht3DU7wpjrJuh6ncLTIb.7.1774272219841.0; ttcsid_CUM82PBC77U4QKJNCRL0=1774267434288::PDIQSPBszIxVOk9hA6pz.7.1774272219841.1"""

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "*/*",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0",
    "Cookie": COOKIES,
    "x-domain": "ugcReview",
    "x-client-interface": "desktop",
    "x-route-prefix": "vi-vn", 
    "Origin": "https://www.traveloka.com",
    "Referer": "https://www.traveloka.com/"
}


def crawl_full_hotel_reviews(hotel_id):
    all_hotel_data = []
    skip = 0
    limit = 40 
    
    while True:
        payload = {
            "fields": [],
            "clientInterface": "desktop",
            "data": {
                "objectId": str(hotel_id),
                "productType": "HOTEL",
                "configId": "REV_CONSV2_HOTEL_GENERAL_V1",
                "filter": {
                    "format": "FORMAT_VALUE_TEXT", 
                    "rating": "RATING_VALUE_ALL", 
                    "language": "LANGUAGE_VALUE_ALL"
                },
                "limit": str(limit),
                "origin": "TRAVELOKA",
                "ratingTagSet": [],
                "skip": str(skip), 
                "sort": "SORT_CREATED_DESCENDING"
            }
        }
        
        try:
            response = requests.post(URL, json=payload, headers=HEADERS, timeout=20)
            
            if response.status_code == 200:
                json_data = response.json().get('data', {})
                reviews = json_data.get('reviews', [])
                has_next = json_data.get('hasNext', False)
                
                if not reviews:
                    break
                
                all_hotel_data.extend(reviews)
                print(f"   -> Đã lấy {len(all_hotel_data)} review (skip={skip})...")
                
                if not has_next:
                    break
                
                skip += limit 
                time.sleep(random.uniform(1.5, 2.5))
            else:
                print(f"   -> Lỗi {response.status_code} tại skip {skip}")
                break
        except Exception as e:
            print(f"   -> Lỗi kết nối: {e}")
            break
            
    return all_hotel_data

if not os.path.exists("./data/id2.csv"):
    print("Vui lòng chạy bước lấy ID khách sạn trước!")
else:
    df_ids = pd.read_csv("./data/id2.csv")
    final_results = []

    print(f"Bắt đầu 'vét sạch' comment cho {len(df_ids)} khách sạn...")

    for index, row in tqdm(df_ids.iterrows(), total=len(df_ids), desc="Tổng tiến độ"):
        h_id = row['Hotel ID']
        h_name = row['Name']
        
        print(f"\n[{index+1}/{len(df_ids)}] Khách sạn: {h_name}")
        
        reviews_list = crawl_full_hotel_reviews(h_id)
        
        for r in reviews_list:
            content = r.get('reviewContentText')
            if content:
                final_results.append({
                    "hotel_id": h_id,
                    "hotel_name": h_name,
                    "comment": content, 
                    "rating": r.get('reviewScore'),
                    "user": r.get('reviewer', {}).get('reviewerName')
                })
        
        time.sleep(3)

    if final_results:
        df_final = pd.DataFrame(final_results)
        # Loại bỏ trùng lặp nếu lỡ cào lặp
        df_final = df_final.drop_duplicates(subset=['comment'])
        
        output_file = "./data/data2_review.csv"
        df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n--- HOÀN THÀNH ---")
        print(f"Tổng cộng đã thu hoạch được {len(df_final)} comment.")
        print(f"File lưu tại: {output_file}")
    else:
        print("\n--- THẤT BẠI: Không lấy được dữ liệu. Hãy kiểm tra lại COOKIE! ---")

Bắt đầu 'vét sạch' comment cho 21 khách sạn...


Tổng tiến độ:   0%|          | 0/21 [00:00<?, ?it/s]


[1/21] Khách sạn: Khách sạn Đồng Khánh
   -> Đã lấy 36 review (skip=0)...


Tổng tiến độ:   5%|▍         | 1/21 [00:03<01:10,  3.53s/it]


[2/21] Khách sạn: Express by M Village Nguyen Du
   -> Đã lấy 17 review (skip=0)...


...


[20/21] Khách sạn: M Village Living Vo Thi Sau
   -> Đã lấy 12 review (skip=0)...


Tổng tiến độ:  95%|█████████▌| 20/21 [01:14<00:03,  3.47s/it]


[21/21] Khách sạn: ibis Sài Gòn Airport
   -> Đã lấy 40 review (skip=0)...
   -> Đã lấy 70 review (skip=40)...


Tổng tiến độ: 100%|██████████| 21/21 [01:20<00:00,  3.84s/it]


--- HOÀN THÀNH ---
Tổng cộng đã thu hoạch được 519 comment.
File lưu tại: ./data/data2_review.csv
